# Temperature

Use ocean fraction on surface temperature to remove land and keep sea surface temperature

In [1]:
import xarray as xr

In [2]:
path_to_members = '/glade/campaign/cgd/ccr/E3SMv2/FV_regridded'
path_to_monthly = '/atm/proc/tseries/month_1/'


# member names
m_names = ['0101', '0111', '0121', '0131', '0141', '0151', '0161',
           '0171', '0181', '0191', '0201', '0211', '0221', '0231',
           '0241', '0251', '0261', '0271', '0281', '0291', '0301']

# member header
m_h = 'v2.FV1.historical_'

# variable
var = 'OCNFRAC'

# member file paths
m_fps = []
for n in m_names:
    file = m_h + n + '.eam.h0.' + var + '.185001-201412.nc'
    file_path = path_to_members + '/' + m_h + n + path_to_monthly + file
    m_fps.append(file_path)
len(m_fps)

21

In [5]:
test = xr.open_dataset(m_fps[0])
test

<xarray.Dataset> Size: 439MB
Dimensions:              (lat: 192, lon: 288, nbnd: 2, cosp_ht: 40,
                          cosp_htmisr: 16, cosp_prs: 7, cosp_reffice: 6,
                          cosp_reffliq: 6, cosp_scol: 10, cosp_sr: 15,
                          cosp_sza: 5, cosp_tau: 7, cosp_tau_modis: 7,
                          ilev: 73, lev: 72, time: 1980)
Coordinates: (12/15)
  * lat                  (lat) float64 2kB -90.0 -89.06 -88.12 ... 89.06 90.0
  * lon                  (lon) float64 2kB 0.0 1.25 2.5 ... 356.2 357.5 358.8
  * cosp_ht              (cosp_ht) float64 320B 1.896e+04 1.848e+04 ... 240.0
  * cosp_htmisr          (cosp_htmisr) float64 128B 0.0 250.0 ... 1.8e+04
  * cosp_prs             (cosp_prs) float64 56B 9e+04 7.4e+04 ... 2.45e+04 9e+03
  * cosp_reffice         (cosp_reffice) float64 48B 5e-06 1.5e-05 ... 7.5e-05
    ...                   ...
  * cosp_sza             (cosp_sza) float64 40B 0.0 20.0 40.0 60.0 80.0
  * cosp_tau             (cosp_tau) float64 56B 0.15 0.8 2.45 ... 41.5 100.0
  * cosp_tau_modis       (cosp_tau_modis) float64 56B 0.15 0.8 ... 41.5 100.0
  * ilev                 (ilev) float64 584B 0.1 0.1477 0.218 ... 997.0 1e+03
  * lev                  (lev) float64 576B 0.1238 0.1828 0.2699 ... 993.8 998.5
  * time                 (time) object 16kB 1850-02-01 00:00:00 ... 2015-01-0...
Dimensions without coordinates: nbnd
Data variables: (12/37)
    lat_bnds             (lat, nbnd) float64 3kB ...
    lon_bnds             (lon, nbnd) float64 5kB ...
    gw                   (lat) float64 2kB ...
    area                 (lat, lon) float64 442kB ...
    P0                   float64 8B ...
    cosp_ht_bnds         (cosp_ht, nbnd) float64 640B ...
    ...                   ...
    nscur                (time) int32 8kB ...
    nsteph               (time) int32 8kB ...
    sol_tsi              (time) float64 16kB ...
    time_bnds            (time, nbnd) object 32kB ...
    time_written         (time) |S8 16kB ...
    OCNFRAC              (time, lat, lon) float32 438MB ...
Attributes: (12/26)
    ne:                        30
    fv_nphys:                  2
    title:                     EAM History file information
    source:                    E3SM Atmosphere Model
    source_id:                 32655baa1f
    product:                   model-output
    ...                        ...
    remap_hostname:            nid006468
    remap_version:             5.0.1
    NCO:                       netCDF Operators version 5.0.1 (Homepage = htt...
    nco_openmp_thread_number:  2
    map_file:                  /global/u2/s/strandwg/proj/proc/2E3SM/map_ne30...
    input_file:                /pscratch/sd/s/strandwg/E3SMv2/v2.LR.historica...

In [6]:
# open files and concat them into single ds (filter data first, large files)
ds = []
for m_fp in m_fps:
    m_ds = xr.open_dataset(m_fp, engine='h5netcdf')
    # select surface level and temperature variable
    m_ds = m_ds['OCNFRAC']
    ds.append(m_ds)
all_ds = xr.concat(ds, dim='member')

In [7]:
all_ds.to_netcdf('/glade/work/acruz/E3SMv2LE/E3SMv2le_OCNFRACs.nc')